In [1]:
!pip install astropy
!pip install lmfit

In [2]:
=0

SyntaxError: invalid syntax (720066093.py, line 1)

In [ ]:
from astropy.io import fits
from matplotlib import pyplot as plt
from astropy.wcs import WCS
from matplotlib.colors import LogNorm
from astropy import constants as const
from astropy import units as u
import pandas as pd
import numpy as np
from astropy.modeling import models


Abrimos el archivo de datos con la imagen de la emisión de las estrellas de la galaxia IRAS09022

In [ ]:
hdul_IRAS09022_R = fits.open('/content/drive/MyDrive/Colab Notebooks/APEX/iras09022_r.fits')

In [ ]:
hdul_IRAS09022_R.info()

In [ ]:
hdul_IRAS09022_R[0].header

In [ ]:
plt.imshow(hdul_IRAS09022_R[0].data)
plt.colorbar()
#

In [ ]:
plt.hist(hdul_IRAS09022_R[0].data.flatten(), bins=50)
plt.show()

In [ ]:
plt.imshow(hdul_IRAS09022_R[0].data, cmap='gist_heat', norm=LogNorm(vmin=5e-2, vmax=10))#show+ scale and color vmin=3.835e-02,vmax=1.24
plt.colorbar()

In [ ]:
plt.imshow(hdul_IRAS09022_R[0].data, cmap='gist_heat', norm=LogNorm(vmin=5e-2, vmax=10))#show+ scale and color vmin=3.835e-02,vmax=1.24
plt.xlim(2000,3000)
plt.ylim(2200,3500)
#plt.colorbar()

In [ ]:
#proyectamos en las coordenadas celestes
wcs= WCS(hdul_IRAS09022_R[0].header)

In [ ]:
plt.figure()
ax = plt.subplot(projection=wcs)
plt.imshow(hdul_IRAS09022_R[0].data, cmap='gist_heat', norm=LogNorm(vmin=5e-2, vmax=10))

plt.xlim(1500,4500)
plt.ylim(1500,4500)

$v=\left(\frac{f_o-f}{f_o}\right)c$

$f_o=\frac{f_{res}}{1+z}$

$f_{res}= 752.033 GHz$

In [ ]:
c=const.c.to(u.km/u.s).value
c

In [ ]:
752.033/(1+0.05964)

In [ ]:
h2o=pd.read_csv('/content/drive/MyDrive/TALLER_PYTHON/epic_spectra.csv')

In [ ]:
h2o

In [ ]:
fig, ax1 = plt.subplots(figsize = (10,6))
ax1.step(h2o['freq'],h2o['flux'],color = 'grey', linewidth= 1, where = 'mid',label='APEX')
plt.fill_between(h2o['freq'],h2o['flux'],step="mid",color='yellow', alpha=0.8)
#ax1.set_xlim([5.4e9, 6e9])
#plt.fill_between_steps(ax1, x,y)
#ax1.set_xlim([-3e9,3e9])
ax1.set_ylim([-0.4, 0.7])

In [ ]:
cube_vel=h2o

In [ ]:
cube_vel

In [ ]:
cube_vel['freq']=((709.705*1e9-h2o['freq'])/(709.705*1e9))*c
#cube_vel

In [ ]:
cube_vel

In [ ]:
g1 = models.Gaussian1D(amplitude=0.45087,mean=108.904, stddev=176.15528663233718)

In [ ]:
g1

In [ ]:
fig, ax1 = plt.subplots(figsize = (10,6))
ax1.step(cube_vel['freq'],cube_vel['flux'],color = 'grey', linewidth= 1, where = 'mid',label='APEX')
plt.fill_between(h2o['freq'],h2o['flux'],step="mid",color='yellow', alpha=0.8)
plt.plot(cube_vel['freq'],g1(cube_vel['freq']),color='red')
#ax1.set_xlim([5.4e9, 6e9])
#plt.fill_between_steps(ax1, x,y)
#ax1.set_xlim([-3e9,3e9])
ax1.set_ylim([-0.4, 0.7])

In [ ]:
from lmfit import Model

In [ ]:
def gaussian(x, amp, cen, wid):
    """1-d gaussian: gaussian(x, amp, cen, wid)"""
    return (amp / (np.sqrt(2*np.pi) * wid)) * np.exp(-(x-cen)**2 / (2*wid**2))

In [ ]:
gmodel = Model(gaussian)

In [ ]:
result_fitting = gmodel.fit(cube_vel["flux"], x=cube_vel["freq"], amp=0.4, cen=108.0, wid=176)

In [ ]:
result_fitting

In [ ]:
result_fitting.best_fit

In [ ]:
fig, ax1 = plt.subplots(figsize = (10,6))
ax1.step(cube_vel['freq'],cube_vel['flux'],color = 'grey', linewidth= 1, where = 'mid',label='APEX')
plt.fill_between(h2o['freq'],h2o['flux'],step="mid",color='yellow', alpha=0.8)
plt.plot(cube_vel["freq"], 1000*result_fitting.best_fit, '-', label='best fit',color='red')
ax1.set_ylim([-0.4, 0.7])